# NIST TN 1822 — Verif.2.9: Group behaviours

15 m x 20 m room. Group 1: 4 agents at 1.25 m/s + 1 agent at 0.5 m/s, in the top zone. Group 2: 10 agents at 0.2 m/s, in the central zone. Group 1 arrivals at the exit should be within 10 s of each other.

Note: the original config had a custom `group_id` per distribution which was stripped (loader has no native group support). Group cohesion here must be verified by the arrival-spread check below; it is NOT enforced by the simulator.

In [1]:
from datetime import datetime
print(f"Executed on {datetime.now().astimezone().strftime('%d %B %Y, %H:%M %Z')}")

Executed on 23 May 2026, 09:46 UTC


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pedpy
from shapely.geometry import Point, Polygon

from jupedsim_scenarios import load_scenario, run_scenario

In [3]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f7f7f5",
    "axes.edgecolor": "#3a3a3a",
    "axes.labelcolor": "#1d1d1d",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.figsize": (8, 5),
})

## Load and run

In [4]:
SCENARIO_ZIP = Path("scenario_files") / "Nist-2-9-groups.zip"
scenario = load_scenario(str(SCENARIO_ZIP))
print(scenario.summary())
result = run_scenario(scenario, seed=42)
df = result.trajectory_dataframe()

Scenario: /work/standards/nist/scenario_files/Nist-2-9-groups.zip
  Model:         CollisionFreeSpeedModel
  Seed:          42
  Max time:      240s
  Exits:         1
  Distributions: 3
  Stages:        0
  Zones:         0
  Journeys:      3
  Agents:        ~15
  Journey elems: 2
  Route:         1 distribution, 0 checkpoint, 1 exit
  Sequence:      jps-distributions_0 -> jps-exits_0
    jps-distributions_0: 4 agents
    jps-distributions_1: 1 agents
    jps-distributions_2: 10 agents
Using fallback logic: No journeys defined
Processing with parameters: {'number': 4, 'radius': 0.15, 'v0': 1.25, 'distribution_mode': 'by_number', 'radius_distribution': 'constant', 'v0_distribution': 'constant', 'use_flow_spawning': False}
Using default parameters: v0=1.25, radius=0.15, n_agents=4

Distribution jps-distributions_0: {'number': 4, 'radius': 0.15, 'v0': 1.25, 'distribution_mode': 'by_number', 'percentage': None, 'use_flow_spawning': False, 'flow_start_time': 0, 'flow_end_time': 10, 'use_p

## Identify Group 1 agents and their arrival times

In [5]:
first = df.sort_values('frame').groupby('id').first()
group1_ids = first[(first.y >= 16.0) & (first.x >= 10.0)].index.tolist()
arrivals = []
for agent_id in group1_ids:
    sub = df[df.id == agent_id].sort_values('frame')
    reached = sub[(sub.y <= 0.3) & (sub.x >= 7.0) & (sub.x <= 8.0)]
    if len(reached):
        arrivals.append(reached.iloc[0].frame / result.frame_rate)
arrivals = np.array(arrivals)
print(f'group 1 size = {len(group1_ids)}; arrivals (s) = {arrivals}')

group 1 size = 5; arrivals (s) = []


## Acceptance

NIST Verif.2.9 requires Group 1 to reach the exit **together** — the arrival spread must be ≤ 10 s. The CollisionFreeSpeedModel has no group-cohesion model, so the 0.5 m/s member lags the 1.25 m/s members and the spread is ~24 s. The test (`test_nist.py`) encodes this criterion as a strict `xfail`; see issue #151.

In [6]:
if len(arrivals) >= 2:
    spread = float(arrivals.max() - arrivals.min())
    print(f'arrival spread = {spread:.2f} s (target <= 10 s)')
else:
    spread = float('nan')
    print('insufficient group 1 arrivals')
result.cleanup()

insufficient group 1 arrivals
